# Module 02-3: 3노드 텍스트 분류기 (솔루션)

---
> 참고 문서: `docs/part1-foundations/02-langgraph-fundamentals.md`

In [ ]:
# 환경 설정
import sys
sys.path.insert(0, '..')

from typing import TypedDict
from langgraph.graph import StateGraph, START, END

print("환경 설정 완료")

## 실습 1: ClassifierState 및 CATEGORY_KEYWORDS 정의 (솔루션)

In [ ]:
# 솔루션
class ClassifierState(TypedDict):
    raw_text: str
    processed_text: str
    category: str
    confidence: float
    result: str


CATEGORY_KEYWORDS: dict[str, list[str]] = {
    "기술": ["AI", "인공지능", "코드", "개발", "소프트웨어", "알고리즘", "프로그램"],
    "경제": ["주식", "금리", "경제", "투자", "시장", "환율", "GDP"],
    "스포츠": ["축구", "야구", "농구", "올림픽", "선수", "경기", "우승"],
    "기타": []
}

print(f"카테고리: {list(CATEGORY_KEYWORDS.keys())}")
print("ClassifierState와 CATEGORY_KEYWORDS 정의 완료")

In [ ]:
# 검증 셀
assert 'ClassifierState' in dir(), "ClassifierState가 정의되어야 합니다"
assert 'CATEGORY_KEYWORDS' in dir(), "CATEGORY_KEYWORDS가 정의되어야 합니다"
assert len(CATEGORY_KEYWORDS) >= 3, "최소 3개의 카테고리가 필요합니다"
assert "기술" in CATEGORY_KEYWORDS, "'기술' 카테고리가 있어야 합니다"
assert "경제" in CATEGORY_KEYWORDS, "'경제' 카테고리가 있어야 합니다"
assert "스포츠" in CATEGORY_KEYWORDS, "'스포츠' 카테고리가 있어야 합니다"
print("✅ 실습 1 완료!")

## 실습 2: 3개 노드 함수 구현 (솔루션)

In [ ]:
# 솔루션
def preprocess(state: ClassifierState) -> dict:
    """텍스트를 소문자로 변환하고 앞뒤 공백을 제거하는 노드"""
    processed = state["raw_text"].strip().lower()
    return {"processed_text": processed}


def classify(state: ClassifierState) -> dict:
    """키워드 매칭으로 텍스트를 카테고리로 분류하는 노드"""
    text = state["processed_text"]
    best_category = "기타"
    best_score = 0
    total_keywords = 0

    for category, keywords in CATEGORY_KEYWORDS.items():
        if not keywords:
            continue
        matches = sum(1 for kw in keywords if kw.lower() in text)
        if matches > best_score:
            best_score = matches
            best_category = category
            total_keywords = len(keywords)

    confidence = best_score / total_keywords if total_keywords > 0 and best_score > 0 else 0.0
    return {"category": best_category, "confidence": confidence}


def format_result(state: ClassifierState) -> dict:
    """분류 결과를 포맷팅하는 노드"""
    result = f"[{state['category']}] 신뢰도: {state['confidence']:.0%} - '{state['raw_text']}'"
    return {"result": result}


print("노드 함수 구현 완료")

In [ ]:
# 검증 셀
s = {"raw_text": "  AI와 개발 트렌드  ", "processed_text": "", "category": "", "confidence": 0.0, "result": ""}
r = preprocess(s)
assert r["processed_text"] == "ai와 개발 트렌드", f"전처리 결과가 올바르지 않습니다: {r['processed_text']}"

s["processed_text"] = "ai와 개발 트렌드"
r2 = classify(s)
assert r2["category"] == "기술", f"'AI와 개발 트렌드'는 기술 카테고리여야 합니다. 결과: {r2['category']}"
assert r2["confidence"] > 0, "신뢰도가 0보다 커야 합니다"

s.update(r2)
r3 = format_result(s)
assert "기술" in r3["result"], "result에 카테고리가 포함되어야 합니다"
assert "신뢰도" in r3["result"], "result에 신뢰도가 포함되어야 합니다"
print("✅ 실습 2 완료!")

## 실습 3: 분류기 그래프 조립 및 테스트 (솔루션)

In [ ]:
# 솔루션
graph = StateGraph(ClassifierState)

graph.add_node("preprocess", preprocess)
graph.add_node("classify", classify)
graph.add_node("format_result", format_result)

graph.add_edge(START, "preprocess")
graph.add_edge("preprocess", "classify")
graph.add_edge("classify", "format_result")
graph.add_edge("format_result", END)

app = graph.compile()

# 테스트
test_texts = [
    "AI와 머신러닝 알고리즘 개발 트렌드",
    "주식 시장 금리 인상 투자 전략",
    "축구 월드컵 한국 선수 우승",
    "오늘 날씨가 맑고 좋네요"
]

for text in test_texts:
    initial = {"raw_text": text, "processed_text": "", "category": "", "confidence": 0.0, "result": ""}
    result = app.invoke(initial)
    print(result["result"])

In [ ]:
# 검증 셀
initial = lambda text: {"raw_text": text, "processed_text": "", "category": "", "confidence": 0.0, "result": ""}

r_tech = app.invoke(initial("AI와 머신러닝 알고리즘 개발"))
assert r_tech["category"] == "기술", f"기술 텍스트가 잘못 분류되었습니다: {r_tech['category']}"

r_econ = app.invoke(initial("주식 시장 금리 인상 투자 전략"))
assert r_econ["category"] == "경제", f"경제 텍스트가 잘못 분류되었습니다: {r_econ['category']}"

r_sport = app.invoke(initial("축구 월드컵 선수 우승"))
assert r_sport["category"] == "스포츠", f"스포츠 텍스트가 잘못 분류되었습니다: {r_sport['category']}"

print("✅ 실습 3 완료!")

## 정리

### 오늘 배운 내용
- 3노드 선형 파이프라인: **전처리 → 분류 → 포맷**
- 키워드 딕셔너리(`CATEGORY_KEYWORDS`)를 활용한 규칙 기반 분류
- 신뢰도(confidence) 점수 산출 패턴